# 🧠 Reasoning Models — GRPO with Verifiable Rewards on Arabic Math

> **HackAI 2026 — Master's track** · Notebook 4/8 · ⏱️ 120 minutes · 🏆 100 pts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

The release of **DeepSeek-R1** in January 2025 changed the rules: a small open model trained with **GRPO** (Group Relative Policy Optimization) on **verifiable rewards** matched o1-preview on math/code. No human preference data, no reward model — just *"is the final answer correct?"*

This notebook teaches you the SOTA recipe end-to-end:

1. The math behind **GRPO** (and why it killed PPO for reasoning)
2. How to design **verifiable reward functions** for Arabic math
3. Train a small Qwen2.5 reasoning model with **TRL ≥ 0.14** in Colab
4. Compare reasoning traces in EN / AR / Darija
5. Evaluate on a held-out Darija math benchmark

### 🏆 Challenge — *"Berhana Math"*
A held-out test set of **50 grade-school Darija/Arabic word problems**. Submit your trained model + Langfuse trace of evaluation. Top accuracy + smallest model wins.

---


## 1 · GRPO in 5 minutes

PPO (the classical RLHF algorithm) needs:
- A **value model** (1 extra network, expensive)
- A **reward model** (trained on human preferences, fragile)
- KL-tracking against the reference model

GRPO (DeepSeek, 2024) drops the value model entirely. The trick: for each prompt, sample $G$ completions, score them, and use the **group-relative advantage**:

$$
A_i = \frac{r_i - \text{mean}(r_{1..G})}{\text{std}(r_{1..G})}
$$

That's it. The advantage is normalized within the group, no critic required. Combined with a verifiable reward (e.g. *"does answer match ground truth?"*), it's enough to bootstrap chain-of-thought reasoning from a base model.

```
Prompt ──▶  sample G=8 completions
            │
            ▼
       reward each  →  r₁ … r₈
            │
            ▼
       A_i = (r_i - μ) / σ          (group-relative advantage)
            │
            ▼
       PPO-style clipped update on the policy, KL-anchored to reference
```

Why this matters in 2026: **every** open reasoning model (DeepSeek-R1, Qwen-QwQ, Llama-Nemotron, Magistral) uses some flavor of GRPO. It's the dominant paradigm.

---


## 2 · Setup

In [ ]:
%pip install -q --upgrade \
    "trl>=0.14" \
    "transformers>=4.50" \
    "peft>=0.14" \
    "datasets>=3.2" \
    "accelerate>=1.4" \
    "bitsandbytes>=0.45" \
    "vllm>=0.7" \
    "math-verify>=0.5" \
    "wandb" "rich"

import os, getpass, torch
if not os.getenv("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")
print("CUDA:", torch.cuda.is_available(), "—", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 3 · Build a Darija/Arabic math dataset

We'll mix three sources:
- **GSM8K-AR** (Arabic translation of GSM8K)
- **Darija-MATH** (synthetic, generated below)
- A few **bilingual** problems for robustness

In [ ]:
# @title Synthesize Darija math problems with templates
import random, json

DARIJA_TEMPLATES = [
    ("3andi {a} d les pommes. 3titek {b}. Cha7al b9a 3andi?",
     lambda a,b: a-b),
    ("Si {name} chra {a} d les cahiers, kol cahier b {b} dirham. Cha7al khlas?",
     lambda a,b,**k: a*b),
    ("F lqa3a kayn {a} d les bnat w {b} d les wlad. Cha7al d les telmid kollchi?",
     lambda a,b: a+b),
    ("Wlad lmadrasa hom {a}. Tlt-hom binat. Cha7al d les binat?",
     lambda a: a//3),
    ("3andi {a} dirham. Bghit nchri ktab b {b}. Cha7al ghadi yb9a liya?",
     lambda a,b: a-b),
]

ARABIC_TEMPLATES = [
    ("لدى أحمد {a} تفاحة. أعطى صديقه {b} منها. كم تبقى لديه؟",
     lambda a,b: a-b),
    ("اشترت فاطمة {a} كتب، ثمن كل كتاب {b} درهم. كم دفعت؟",
     lambda a,b: a*b),
    ("في الفصل {a} طالبًا و{b} طالبة. كم عدد التلاميذ؟",
     lambda a,b: a+b),
]

NAMES = ["Hmed","Karim","Fatima","Salma","Youssef","Adam","Zineb"]
random.seed(42)

def gen(n=200):
    out = []
    for _ in range(n):
        if random.random() < 0.6:
            tpl, fn = random.choice(DARIJA_TEMPLATES)
            lang = "darija"
        else:
            tpl, fn = random.choice(ARABIC_TEMPLATES)
            lang = "arabic"
        params = {"a": random.randint(5, 100), "b": random.randint(2, 30),
                  "name": random.choice(NAMES)}
        try:
            answer = fn(**{k:v for k,v in params.items() if k in fn.__code__.co_varnames})
        except Exception:
            continue
        if answer < 0: continue
        q = tpl.format(**params)
        out.append({"problem": q, "answer": str(answer), "lang": lang})
    return out

problems = gen(300)
print(f"Generated {len(problems)} problems")
for p in problems[:5]: print(p)

In [ ]:
# @title Wrap as a HF Dataset and split
from datasets import Dataset
ds = Dataset.from_list(problems)
ds = ds.train_test_split(test_size=0.1, seed=42)
print(ds)

## 4 · The reward function — the heart of GRPO

A reward function is a Python callable that takes a list of generated completions and returns a list of floats. The closer to "verifiable" (i.e. *programmatically checkable*), the better. We use two:

- `correctness_reward` — does the boxed answer match ground truth?
- `format_reward` — did the model emit the `<think>...</think>` and `\boxed{...}` structure?

In [ ]:
# @title Reward functions
import re
from math_verify import parse, verify

THINK_RE   = re.compile(r"<think>.*?</think>", re.DOTALL)
ANSWER_RE  = re.compile(r"\\boxed\{([^}]+)\}")

def extract_answer(text: str) -> str | None:
    m = ANSWER_RE.search(text)
    return m.group(1).strip() if m else None

def correctness_reward(prompts, completions, answer, **kw):
    """Reward 1.0 if the boxed answer is numerically equal to ground truth."""
    rewards = []
    for c, gt in zip(completions, answer):
        pred = extract_answer(c[0]["content"] if isinstance(c, list) else c)
        if pred is None:
            rewards.append(0.0); continue
        try:
            ok = verify(parse(f"$\boxed{{{pred}}}$"), parse(f"$\boxed{{{gt}}}$"))
            rewards.append(1.0 if ok else 0.0)
        except Exception:
            rewards.append(1.0 if str(pred).strip() == str(gt).strip() else 0.0)
    return rewards

def format_reward(prompts, completions, **kw):
    """Reward 0.5 for proper <think> + boxed structure (independent of correctness)."""
    rewards = []
    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else c
        has_think = bool(THINK_RE.search(text))
        has_box   = bool(ANSWER_RE.search(text))
        rewards.append(0.5 * (int(has_think) + int(has_box)) / 2)
    return rewards

# Quick sanity check
fake_completion = "<think>3+5=8</think>\n\\boxed{8}"
print("correct:", correctness_reward(["x"], [fake_completion], ["8"]))
print("format :", format_reward(["x"], [fake_completion]))

## 5 · System prompt that scaffolds reasoning

In [ ]:
SYSTEM_PROMPT = """You are a careful math tutor who reasons step-by-step in the SAME language as the question (Darija, Arabic, or English).

Your output MUST follow this exact format:

<think>
Step-by-step reasoning here. Show every calculation.
</think>
\\boxed{FINAL_ANSWER}

The boxed answer must be a single number, no units, no extra text."""

def to_chat(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": example["problem"]},
        ],
        "answer": example["answer"],
    }

train = ds["train"].map(to_chat)
test  = ds["test"].map(to_chat)
print(train[0])

## 6 · GRPO training with TRL

Even on a free Colab T4 you can train a 0.5B Qwen with LoRA + GRPO. For full-finetune of 1.5–7B you'll want an A100.

In [ ]:
# @title Tiny GRPO training run (T4-friendly)
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig

base = "Qwen/Qwen2.5-0.5B-Instruct"
tok  = AutoTokenizer.from_pretrained(base)

policy = AutoModelForCausalLM.from_pretrained(
    base, torch_dtype=torch.bfloat16, device_map="auto"
)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    task_type="CAUSAL_LM",
)

cfg = GRPOConfig(
    output_dir="grpo-darija-math",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=4,           # G in the formula above
    max_prompt_length=512,
    max_completion_length=512,
    logging_steps=5,
    save_steps=200,
    bf16=True,
    report_to="none",            # set to "wandb" if you wire it
    beta=0.04,                   # KL coefficient
)

trainer = GRPOTrainer(
    model=policy,
    processing_class=tok,
    reward_funcs=[correctness_reward, format_reward],
    args=cfg,
    train_dataset=train.select(range(50)),  # small for the demo
    peft_config=lora,
)

# Uncomment to actually train (~15 min on T4 for 50 steps)
# trainer.train()
print("✅ Trainer ready. Uncomment trainer.train() to start.")
print("   Recommended: run on the full train set on an A100 for ~30 min.")

## 7 · Inference with `<think>` tags

After training, your model spontaneously emits chain-of-thought between `<think>` tags. You can choose to *show* or *hide* them in the UX (DeepSeek-R1 style).

In [ ]:
# @title Generate a reasoning trace
from transformers import pipeline

# In production, swap with the trained checkpoint path
model_id = base  # "grpo-darija-math/checkpoint-XXX"
gen = pipeline("text-generation", model=model_id, torch_dtype=torch.bfloat16,
               device_map="auto", tokenizer=tok)

prompt = tok.apply_chat_template([
    {"role":"system","content": SYSTEM_PROMPT},
    {"role":"user","content": "Si Hmed 3andou 45 dirham, chra cahier b 12 w stylo b 8. Cha7al b9a 3andou?"},
], tokenize=False, add_generation_prompt=True)

out = gen(prompt, max_new_tokens=400, do_sample=False)[0]["generated_text"]
# Show only the part after the prompt
print(out[len(prompt):])

## 8 · Evaluation harness (use as your leaderboard submission)

In [ ]:
# @title Evaluate on the test split
import time

def evaluate(generate_fn, dataset, n=50):
    n = min(n, len(dataset))
    correct, t0 = 0, time.time()
    for ex in dataset.select(range(n)):
        prompt = tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True)
        out = generate_fn(prompt)
        pred = extract_answer(out)
        if pred and str(pred).strip() == str(ex["answer"]).strip():
            correct += 1
    return {
        "accuracy": correct / n,
        "n": n,
        "seconds_per_example": (time.time() - t0) / n,
    }

def gen_fn(p):
    return gen(p, max_new_tokens=400, do_sample=False)[0]["generated_text"][len(p):]

print(evaluate(gen_fn, test, n=10))   # bump to 50 for the full eval

## 9 · 🏆 Challenge — *"Berhana Math"*

Your submission must include:
1. Trained model on the HF Hub (any size up to 7B)
2. Eval script reproducing your accuracy
3. **Langfuse trace** of evaluation showing 50 problems
4. A short README on **what reward shaping** you tried

### Bonus categories
- 🥇 **Most efficient**: highest accuracy / parameter-count
- 🎯 **Most multilingual**: best Darija + Arabic + English balance
- 🧪 **Best ablation**: most informative reward-function comparison

### Tips for pushing accuracy
- **Mix reward signals**: correctness (1.0) + format (0.5) + brevity (0.1 penalty per token over 256)
- **Curriculum**: start with single-step problems, escalate to multi-step
- **Self-consistency at inference**: sample 4 traces, majority-vote the boxed answer
- **Better base**: Qwen2.5-Math-1.5B already knows arithmetic; GRPO teaches it to *show its work*

---


## 10 · Recap

You learned:
- ✅ Why **GRPO** replaced PPO as the reasoning-RL algorithm of choice
- ✅ How to design **verifiable rewards** that bootstrap CoT
- ✅ How to train a small Qwen with **TRL + LoRA** on Colab
- ✅ How to evaluate Darija/Arabic math reasoning end-to-end

**Next →** `05_rag_arabic.ipynb`: RAG that actually works on Arabic — embeddings, hybrid search, re-ranking.

> *"The biggest reasoning gain since chain-of-thought prompting."* — DeepSeek-R1 paper, on GRPO + verifiable rewards.
